# GOES Data
This noteboook downloads data from GOES.

### Notes:
- Cleaning step: the > 0 filter drops negative fill values and zeros. If you want the raw download exactly as served, comment out that line, though then drop_duplicates and sorting are still worth keeping.
- Empty responses: if HAPI has no data for the interval, raise_for_status() may raise an HTTP error rather than returning an empty table, so check the interval and dataset ID if that happens.
- The while loop is a 7-day chunking loop, which will allow this code to be used for longer time spans without the server timing out.

In [ ]:
import io
import os
import json
import requests
import pandas as pd
from datetime import timedelta

In [ ]:
# ---------------- User settings ----------------
start = pd.Timestamp("2022-03-18 18:00:00", tz="UTC")
end   = pd.Timestamp("2022-03-19 18:00:00", tz="UTC")

output_folder = r"C:\Users\pauls\SEPs\coseecat_dr1\data\19032022\GOES"
os.makedirs(output_folder, exist_ok=True)

BASE = "https://iswa.ccmc.gsfc.nasa.gov/hapi"
dataset_id = "goesp_part_flux_P5M"
# -----------------------------------------------

In [ ]:
# 1. Get dataset metadata (column names, units, fill values)
r = requests.get(f"{BASE}/info", params={"id": dataset_id}, timeout=60)
r.raise_for_status()
info = r.json()

columns = []     # column names in the order the server returns them
fills = {}       # fill values to be converted to NaN

for p in info["parameters"]:
    n = 1
    for s in p.get("size", [1]):
        n *= s
    cols = [p["name"]] if n == 1 else [f"{p['name']}_{i}" for i in range(n)]
    columns += cols

    if p["name"] == "Time":
        continue

    # HAPI stores fill as a string; "null" / None means none is declared
    try:
        fv = float(p.get("fill"))
    except (TypeError, ValueError):
        fv = None

    if fv is not None:
        for c in cols:
            fills[c] = fv

print("Columns available:")
for p in info["parameters"]:
    print(f"  {p['name']:<15} units={p.get('units')}  "
          f"description={p.get('description')}")

In [ ]:
# 2. Download (same 7-day chunking as before)
def get_iswa_data(start, end):
    data = []

    while start < end:
        time_interval_end = min(start + timedelta(days=7), end)

        params = {
            "id": dataset_id,
            "time.min": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "time.max": time_interval_end.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "format": "csv",          # no "parameters" -> all channels
        }

        r = requests.get(f"{BASE}/data", params=params, timeout=120)
        r.raise_for_status()

        df = pd.read_csv(io.StringIO(r.text), comment="#", names=columns)
        data.append(df)

        start = time_interval_end

    df = pd.concat(data, ignore_index=True)
    df["Time"] = pd.to_datetime(df["Time"], utc=True)

    for c in columns[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Replace the server's fill values with NaN (keeps all rows/channels)
    for c, fv in fills.items():
        df.loc[df[c] == fv, c] = float("nan")

    return df.drop_duplicates("Time").sort_values("Time").reset_index(drop=True)


df = get_iswa_data(start, end)

if df.empty:
    raise ValueError("No data returned for this interval.")



In [ ]:
# 3. Save data and metadata
stamp = f"{start.strftime('%Y%m%dT%H%M')}_to_{end.strftime('%Y%m%dT%H%M')}"
csv_path  = os.path.join(output_folder, f"GOES_all_channels_{stamp}.csv")
meta_path = os.path.join(output_folder, f"GOES_all_channels_{stamp}_info.json")

df.to_csv(csv_path, index=False)
with open(meta_path, "w") as f:
    json.dump(info, f, indent=2)

print(f"\nDownloaded {len(df)} rows x {len(df.columns)} columns")
print(f"Time range: {df['Time'].min()} to {df['Time'].max()}")
print(f"Saved data to:     {os.path.abspath(csv_path)}")
print(f"Saved metadata to: {os.path.abspath(meta_path)}")